In [1]:
import pandas as pd

In [ ]:
import numpy as np

individual_sample = pd.read_csv('randomsample_part2_with_predictions.csv')

print(individual_sample.head())

   eroll_no constituency_name  gender age_band area_type  R    G   SG  \
0       190         Chunchura    Male    26-35       Rur  H  GEN  MAT   
1         8          Natabari    Male      60+       Rur  M  GEN   BM   
2       133             Kulpi    Male    36-45       Urb  H  GEN  GUC   
3        41       Gangarampur    Male    36-45       Rur  M  GEN   BM   
4       241  Joypur (Purulia)  Female      60+       Rur  H  OBC  MAH   

   n_individuals  weighted_n  stratum_share  
0              4    0.249302       0.000938  
1             38    2.758694       0.011311  
2             12    0.522698       0.002124  
3             51    4.322048       0.017404  
4            107    9.095732       0.038503  


In [3]:
candidates = pd.read_csv('west_bengal_candidates_2026.csv')
candidates.head()

,voting_date,district,constituency_no,constituency_name,AITC+_Party,AITC+_Candidate,NDA_Party,NDA_Candidate,LF+_Party,LF+_Candidate
0,23 April 2026,Cooch Behar district,1,Mekliganj,Trinamool Congress,Paresh Chandra Adhikary,Bharatiya Janata Party,Dadhiram Roy,All India Forward Bloc,Kamal Roy
1,23 April 2026,Cooch Behar district,2,Mathabhanga,Trinamool Congress,Sablu Barman,Bharatiya Janata Party,Nisith Pramanik,Communist Party of India (Marxist),Khagen Barman
2,23 April 2026,Cooch Behar district,3,Cooch Behar Uttar,Trinamool Congress,Partha Pratim Roy,Bharatiya Janata Party,Sukumar Roy,Communist Party of India (Marxist),Pranay Karji
3,23 April 2026,Cooch Behar district,4,Cooch Behar Dakshin,Trinamool Congress,Avijit Dey Bhowmik,Bharatiya Janata Party,Rathindra Nath Bose,All India Forward Bloc,Nazmul Alam Sarkar
4,23 April 2026,Cooch Behar district,5,Sitalkuchi,Trinamool Congress,Harihar Das,Bharatiya Janata Party,Savitri Barman,Communist Party of India (Marxist),Nabadipti Adhikari


In [ ]:
import os

os.environ['SARVAM_API_KEY'] = "insert_your_api_key_here"
API_KEY = os.environ['SARVAM_API_KEY']

In [20]:
import pandas as pd
import requests
import json
import re
import time

# ── API config ─────────────────────────────────────────────────────────────────
url = "https://api.sarvam.ai/v1/chat/completions"

# ── Decode maps ────────────────────────────────────────────────────────────────
R_MAP   = {"H": "Hindu", "M": "Muslim", "O": "Other", "U": "Unknown"}
G_MAP   = {"GEN": "General", "SC": "Scheduled Caste", "ST": "Scheduled Tribe",
           "OBC": "Other Backward Class", "U": "Unknown"}
SG_MAP  = {
    "GUC": "General Upper Caste Hindu (Brahmin / Kayastha / Baidya)",
    "MAT": "Matua / Namasudra Hindu",
    "RAJ": "Rajbangshi Hindu",
    "NAM": "Namasudra (other) Hindu",
    "MAH": "Mahishya Hindu",
    "SAN": "Santhal Adivasi Hindu",
    "TEA": "Tea Garden Worker Hindu",
    "O":   "Other Hindu",
    "BM":  "Bengali Muslim",
    "UM":  "Urdu-speaking Muslim",
    "U":   "Unknown subgroup",
}
AREA_MAP = {"Urb": "Urban", "Rur": "Rural"}

# ── Improved system prompt ─────────────────────────────────────────────────────
# ── Improved system prompt ─────────────────────────────────────────────────────
VOTING_SYSTEM_PROMPT = """You are a West Bengal voter behavior analyst. You will simulate being a real voter and think 
through the decision process.

STEP 1 — Think inside <think> tags as if you ARE this voter. Reason in first person:
  - What is your community's relationship with each party?
  - Read manifesto promises from each party - what are they offering to you?
  - Check on internet what is happening — recent news, social media trends, viral videos, what people are talking about in your locality
  - What have you seen in online news, social media propaganda, and what do you think of it?
  - What do you personally care about — welfare schemes, jobs, safety, identity, land?
  - Who do local leaders in your area support? What are they saying?
  - Are there any community-specific pulls (e.g. CAA for Matua, social welfare schemes for specific communities, mosque/mandir politics)?
  - How has the ruling party treated your community in recent years — development, discrimination, or neglect?
  - Are you a loyal voter (always vote for one party) or persuadable based on this election's issues?

STEP 2 — After your reasoning, output a JSON object with your final decision:
{
  "predicted_vote": "Party Name - Candidate Name",
  "reasoning": "Brief summary of your key deciding factor",
  "confidence": "High/Medium/Low",
  "key_issues": ["issue1", "issue2"],
  "persuadable": true/false
}

Consider the voter's demographic profile and available candidates. If no candidate from a party exists, note that.
Be realistic - consider real West Bengal political dynamics, caste calculations, and welfare politics.

Return ONLY valid JSON after </think>. No extra text. """

# ── Helper: get candidates for a constituency ──────────────────────────────────
def get_candidates(constituency: str, candidates_df: pd.DataFrame) -> dict:
    match = candidates_df[
        candidates_df["constituency_name"].str.strip().str.lower() == constituency.strip().lower()
    ]
    if match.empty:
        return {"error": True, "voting_date": "Unknown",
                "AITC+": "No candidate", "NDA": "No candidate", "LF+": "No candidate"}
    r = match.iloc[0]

    def fmt(p_col, c_col):
        p = r.get(p_col, ""); c = r.get(c_col, "")
        p = "" if pd.isna(p) else str(p).strip()
        c = "" if pd.isna(c) else str(c).strip()
        return f"{p} — {c}" if (p or c) else "No candidate"

    return {
        "error": False,
        "voting_date": r.get("voting_date", "Unknown"),
        "AITC+": fmt("AITC+_Party", "AITC+_Candidate"),
        "NDA":   fmt("NDA_Party",   "NDA_Candidate"),
        "LF+":   fmt("LF+_Party",   "LF+_Candidate"),
    }


# ── Helper: build the user prompt ─────────────────────────────────────────────
def build_prompt(row: pd.Series, cands: dict) -> str:
    religion = R_MAP.get(str(row.get("R", "U")), "Unknown")
    subgroup = SG_MAP.get(str(row.get("SG", "U")), "Unknown")
    const    = row.get("constituency_name", "Unknown")

    return f"""YOU ARE THIS VOTER
──────────────────
Constituency : {const}
Election Date: {cands['voting_date']}
Gender       : {row.get('gender', 'Unknown')}
Age Band     : {row.get('age_band', 'Unknown')}
Area         : {AREA_MAP.get(str(row.get('area_type', '')), row.get('area_type', 'Unknown'))}
Religion     : {religion}
Caste Group  : {G_MAP.get(str(row.get('G', 'U')), 'Unknown')}
Community    : {subgroup}

YOUR CANDIDATES IN {const.upper()}
  • AITC+ : {cands['AITC+']}
  • NDA   : {cands['NDA']}
  • LF+   : {cands['LF+']}

Think as this person, then give your JSON prediction."""


# ── Core API call ──────────────────────────────────────────────────────────────
def predict_vote(row: pd.Series, candidates_df: pd.DataFrame) -> tuple[dict, str | None]:
    cands = get_candidates(row.get("constituency_name", ""), candidates_df)
    if cands["error"]:
        return {"error": "constituency_not_found"}, None

    data = {
        "model": "sarvam-m",
        "messages": [
            {"role": "system", "content": VOTING_SYSTEM_PROMPT},
            {"role": "user",   "content": build_prompt(row, cands)},
        ],
    }

    try:
        resp = requests.post(
            url,
            headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
            json=data,
            timeout=60,
        )
    except requests.exceptions.RequestException as e:
        return {"error": f"request_failed: {e}"}, None

    if resp.status_code != 200:
        print(f"  [HTTP {resp.status_code}] {resp.text[:300]}")
        return {"error": f"http_{resp.status_code}"}, None

    resp_json = resp.json()
    if "choices" not in resp_json:
        print(f"  [BAD RESPONSE] {json.dumps(resp_json)[:300]}")
        return {"error": "no_choices"}, None

    raw = resp_json["choices"][0]["message"]["content"].strip()

    think_match = re.search(r"<think>(.*?)</think>", raw, flags=re.DOTALL)
    thought  = think_match.group(1).strip() if think_match else None
    cleaned  = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()

    json_match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if not json_match:
        return {"error": "no_json", "raw": cleaned}, thought

    try:
        return json.loads(json_match.group()), thought
    except json.JSONDecodeError:
        return {"error": "parse_failed", "raw": cleaned}, thought


# ── Initialise output columns on individual_sample ────────────────────────────
for col in ["predicted_vote", "vote_confidence", "vote_reasoning", "vote_thought"]:
    if col not in individual_sample.columns:
        individual_sample[col] = None

# ── Main loop ──────────────────────────────────────────────────────────────────
total = len(individual_sample)
print(f"Processing {total} voters (~{total * 1.2 / 60:.0f} min estimated)\n{'='*70}")

start_row = 9557 # 1-based index as per your request

for k, (index, row) in enumerate(individual_sample.iloc[start_row-1:].iterrows(), start=start_row-1):
    result, thought = predict_vote(row, candidates)

    # ── Write back to individual_sample ───────────────────────────────
    individual_sample.at[index, "predicted_vote"]   = result.get("predicted_vote")
    individual_sample.at[index, "vote_confidence"]  = result.get("confidence")
    individual_sample.at[index, "vote_reasoning"]   = result.get("reasoning")
    individual_sample.at[index, "vote_thought"]     = thought

    # ── Console output ────────────────────────────────────────────────
    const  = row.get("constituency_name", "?")
    sg     = SG_MAP.get(str(row.get("SG", "U")), "?")
    gender = row.get("gender", "?")
    age    = row.get("age_band", "?")

    print(f"[{k}/{total}] {const} | {gender} {age} | {sg}")
    if "error" in result:
        print(f"  ✗ Error: {result['error']}")
    else:
        print(f"  → {result.get('predicted_vote')}  [{result.get('confidence')}]")
        print(f"  {result.get('reasoning', '')}")
    print()


# ── Quick vote-share summary ───────────────────────────────────────────────────
print(f"\n{'='*70}")
print("VOTE DISTRIBUTION (predicted)")
print("-"*40)
vote_col = individual_sample["predicted_vote"].dropna()
alliance_counts = vote_col.str.extract(r'^(AITC\+|NDA|LF\+|UNDECIDED|NOTA)')[0].value_counts()
total_pred = alliance_counts.sum()
for alliance, cnt in alliance_counts.items():
    print(f"  {alliance:<12}: {cnt:>5}  ({cnt/total_pred*100:.1f}%)")

print(f"\n✓ All results written to individual_sample dataframe.")
print(f"  Columns added: predicted_vote, vote_confidence, vote_reasoning, vote_thought")

Processing 12500 voters (~250 min estimated)
[9556/12500] Howrah Uttar | Female 46-60 | Matua / Namasudra Hindu
  → AITC+ - Trinamool Congress - Gautam Chowdhuri  [Medium]
  As a Scheduled Caste Matua/Namasudra voter, TMC's welfare schemes (pensions, housing) and strong local presence outweigh BJP's Hindutva focus. Dissatisfaction with CPI(M)'s limited SC outreach. TMC's Gautam Chowdhuri has community trust in Howrah Uttar.

[9557/12500] Khargram | Female 18-25 | Matua / Namasudra Hindu
  → AITC+ : Trinamool Congress — Ashis Marjit  [Medium]
  As a young SC woman in rural Khargram, Trinamool's focus on welfare schemes like Kanyashree Prakalpa and local infrastructure development resonates. Strong local leader support and existing beneficiary networks outweigh BJP's CAA emphasis (less relevant to Hindu SC concerns) and CPI-M's diminished grassroots presence.

[9558/12500] Dum Dum Uttar | Female 26-35 | General Upper Caste Hindu (Brahmin / Kayastha / Baidya)
  ✗ Error: no_json

[9559/125

In [21]:
individual_sample.to_csv('randomsample_part2_with_predictions.csv', index=False)